# IMPORT & INSPECT DATASETS

In [ ]:
import pandas as pd  
import numpy as np  
import matplotlib.pyplot as plt  
import seaborn as sns  
from sklearn.model_selection import train_test_split  
from sklearn.preprocessing import StandardScaler

In [ ]:
def inspect_df(df):
    print("\n Data Overview")
    print(df.info())
    print("\n Quick Stats")
    print(df.describe())
    print(f"Data Types:\n{df.dtypes}")
    print(f"Shape: {df.shape}")
    print(f"Columns: {list(df.columns)}")
    print("\n Missing Values")
    null_counts = df.isnull().sum()
    total_nulls = null_counts.sum()
    total_elements = df.size
    null_percentage = (total_nulls / total_elements) * 100
    print(f"Null Values:\n{null_counts}")
    print(f"Total Null Values: {total_nulls}")
    print(f"Total Elements: {total_elements}")
    print(f"Percentage of Null Values: {null_percentage:.2f}%")
    print("\n Descriptive Statistics")
    print(df.describe(include='all'))
    print("\n Duplicate Rows")
    print(f"Total Duplicates: {df.duplicated().sum()}")
    print("\n Value Counts (for categorical columns):")
    for col in df.select_dtypes(include=['object', 'category']).columns:
        print(f"\n{col} - Value Counts:")
        print(df[col].value_counts())

    return df.head()

In [ ]:
df1 = pd.read_csv("Dataset_1960.csv") # Meteorite Data 1960's
df2 = pd.read_csv('Dataset_1970.csv') # Meteorite Data 1970's
df3 = pd.read_csv('Dataset_1980.csv') # Meteorite Data 1980's
df4 = pd.read_csv('Dataset_1990.csv') # Meteorite Data 1990's
df5 = pd.read_csv('Dataset_2000.csv') # Meteorite Data 2000's
df6 = pd.read_csv('Dataset_2010.csv') # Meteorite Data > 2010's
df7 = pd.read_csv('Dataset_Area.csv') # Country & Continent Geographical Data

In [ ]:
inspect_df(df1)

In [ ]:
inspect_df(df2)

In [ ]:
inspect_df(df3)

In [ ]:
inspect_df(df4)

In [ ]:
inspect_df(df5)

In [ ]:
inspect_df(df6)

In [ ]:
df1.drop(columns=['Abbrev','Fall','MetBull','Antarctic'], inplace=True)
df2.drop(columns=['Abbrev','Fall','MetBull','Antarctic'], inplace=True)
df3.drop(columns=['Abbrev','Fall','MetBull','Antarctic'], inplace=True)
df4.drop(columns=['Abbrev','Fall','MetBull','Antarctic'], inplace=True)
df5.drop(columns=['Abbrev','Fall','MetBull','Antarctic'], inplace=True)
df6.drop(columns=['Abbrev','Fall','MetBull','Antarctic'], inplace=True)

In [ ]:
df1.columns = pd.Series(df1.columns).apply(lambda col: col.lower())
df2.columns = pd.Series(df2.columns).apply(lambda col: col.lower())
df3.columns = pd.Series(df3.columns).apply(lambda col: col.lower())
df4.columns = pd.Series(df4.columns).apply(lambda col: col.lower())
df5.columns = pd.Series(df5.columns).apply(lambda col: col.lower())
df6.columns = pd.Series(df6.columns).apply(lambda col: col.lower())
df7.columns = pd.Series(df7.columns).apply(lambda col: col.lower())

In [ ]:
df1.shape

# PREPARE DATASETS 1-6 FOR REVERSE_GEOCODER API (ran in Google Colab due to CPU usage)

### Clean Coordinates column and split into Latitude & Longitude For Country Coding - Marked Down

import pandas as pd

Remove spaces, parentheses, and split into lat/long
df['(lat,long)'] = df['(lat,long)'].str.strip().str.replace(r'[\(\)]', '', regex=True)
df[['Latitude', 'Longitude']] = df['(lat,long)'].str.split(', ', expand=True)

Convert to float
df['Latitude'] = df['Latitude'].astype(float)
df['Longitude'] = df['Longitude'].astype(float)

Drop the original column if needed
df.drop(columns=['(lat,long)'], inplace=True)

Marked Down so it the code doesnt run. Output is Dataset 8

### Retrieve Country Codes - Marked Down

pip install swifter

pip install pandas reverse_geocoder pycountry geopy

import reverse_geocoder as rg
import pycountry
import swifter  # Enables parallel processing

import reverse_geocoder as rg
import pycountry
from tqdm import tqdm

Enable progress bar
tqdm.pandas()

Function to get country code from latitude & longitude
def get_country(row):
    try:
        latitude = float(row['Latitude']) if pd.notna(row['Latitude']) else 0
        longitude = float(row['Longitude']) if pd.notna(row['Longitude']) else 0
        if latitude == 0 and longitude == 0:
            return "N/A"  # Assign placeholder for oceanic locations
        result = rg.search((latitude, longitude))  # Returns a list of dicts
        return result[0]['cc']  # Extract country code
    except Exception as e:
        print(f"Error processing row {row}: {e}")
        return "Unknown"  # Ensure no NaN values are introduced

Function to convert country code to full country name
def country_name(code):
    try:
        return pycountry.countries.get(alpha_2=code).name
    except:
        return "Ocean/Unknown" if code == "N/A" else "Unknown"

Ensure numeric lat/lon without introducing NaNs
df_all['Latitude'] = pd.to_numeric(df_all['Latitude'], errors='coerce').fillna(0)
df_all['Longitude'] = pd.to_numeric(df_all['Longitude'], errors='coerce').fillna(0)

Apply function with a progress bar
df_all['country_code'] = df_all.progress_apply(get_country, axis=1)

Convert country codes to full country names
df_all['country_name'] = df_all['country_code'].apply(country_name)

# CONCATENATE DATASETS 1-6 INTO DATASET 8 + WRANGLING

In [ ]:
df = pd.concat([df1,df2,df3,df4,df5,df6], axis=0)
df

In [ ]:
df.shape

In [ ]:
df.dtypes

In [ ]:
# Drop NULL Values
df['year'] = df['year'].replace('', pd.NA)
df = df.dropna(subset=['year'])
df.shape

In [ ]:
df.columns

In [ ]:
df.drop(columns=['unnamed: 11','unnamed: 12'], inplace=False)

In [ ]:
# Add a decade to the year
def get_decade(year):
    return (year // 10) * 10

# Apply the function to create a new column 'decade'
df['decade'] = df['year'].apply(get_decade)

In [ ]:
df.head(5)

In [ ]:
df['year'] = df['year'].astype(int)
df['decade'] = df['decade'].astype(int)
print(df.dtypes)  # Check data types

In [ ]:
df.head(5)

In [ ]:
# Create a function to inspect the null values and predicted shape of new Dataset
data=df
def null_values(data):
    shape = data.shape[0]
    null_values = data['decade'].isnull().sum()
    perc = data['decade'].isnull().sum()/data.shape[0]*100
    pred_size = data.shape[0] - data['decade'].isnull().sum()
    print('The DataFrame has:')
    print(f'Total Rows: {shape}')
    print(f'Total Null Values: {null_values}')
    print(f'Percentage Null Values: {perc:.2f}%')
    print(f'A predicted size of: {pred_size}')
    return

In [ ]:
null_values(data)

# ADD DATASET 7 (LANDMASS & CONTINENT) TO DATASET 8. CREATED NEW DATAFRAME DF_FULL

In [ ]:
df7 = df7.drop(columns=['slug','date_of_information','ranking'], inplace=False)
df7.rename(columns={'name':'country'}, inplace=True)
df7.head(5)

In [ ]:
df8 = pd.read_csv('Datasheet_All.csv') # df with country data
df8

In [ ]:
df8.rename(columns={'Latitude': 'latitude','Longitude':'longitude','country_code':'ccode','country_name':'country'}, inplace=True)
df8.drop(columns=['unnamed: 11','unnamed: 12'], inplace=True)
df8.head(5)

In [ ]:
df_full = df8.merge(df7, on='country', how='left')
df_full

In [ ]:
df_full.to_csv('Full_Dataset.csv', index = False)

In [ ]:
df_full.dropna()

# MACHINE LEARNING TO STANDARDIZE THE ENTRIES IN THE MASS COLUMN (MARKED DOWN)

## ML Reworked so that NaN values are excluded from the model

In [ ]:
import pandas as pd
import re
from tqdm import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
import numpy as np

tqdm.pandas()

def extract_mass(value):
    if pd.isna(value):
        return None  # Handle missing values
    
    value = str(value).strip().lower()
    match = re.search(r"([\d\.]+)\s*([a-zA-Z]*)", value)
    if not match:
        return None
    
    num, unit = match.groups()
    try:
        num = float(num)
    except ValueError:
        return None

    conversion_factors = {
        "g": 1, "gram": 1, "grams": 1,
        "kg": 1000, "kilogram": 1000, "kilograms": 1000,
        "t": 1000000, "ton": 1000000, "tonne": 1000000, "tonnes": 1000000
    }

    return num * conversion_factors.get(unit, np.nan)  # Default to NaN if unit unknown

print(" Extracting clean mass values...")
df_full["mass_cleaned"] = df_full["mass"].progress_apply(extract_mass)

df_train = df_full.dropna(subset=["mass_cleaned"]).copy()

if len(df_train) < 10:
    print("Not enough valid data for training! Consider manually checking your dataset.")
else:
    print("Extracting text features using TF-IDF...")
    vectorizer = TfidfVectorizer() #converts text data into a numerical representation based on the importance of words
    X = vectorizer.fit_transform(df_train["mass"].astype(str))
    y = df_train["mass_cleaned"]

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    print("Training Random Forest model...")
    model = RandomForestRegressor(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    accuracy = np.mean((abs(y_test - y_pred) / y_test) < 0.1) * 100  # % within 10% error

    print(f"Model trained! Mean Absolute Error: {mae:.2f} grams")
    print(f"R² Score: {r2:.2f}")
    print(f"Percentage of Predictions within 10% Error: {accuracy:.2f}%")

    print("Predicting missing mass values...")
    missing_mask = df_full["mass_cleaned"].isna() & df_full["mass"].notna()
    df_full.loc[missing_mask, "mass_cleaned"] = model.predict(vectorizer.transform(df_full.loc[missing_mask, "mass"].astype(str)))

    df_full["mass_cleaned"] = df_full["mass_cleaned"].round()

df_full.loc[df_full["mass"].isna(), "mass_cleaned"] = np.nan

print("Data processing complete! Here's a sample of the cleaned data:")
print(df_full[["mass", "mass_cleaned"]].head())


# CREATE NEW DATASET DF9

In [ ]:
df9 = pd.read_csv('df_full_ml.csv')

In [ ]:
inspect_df(df9)

In [ ]:
df9.drop(columns=['mass'], inplace=True)

In [ ]:
df9.rename(columns={'mass_cleaned': 'mass'}, inplace=True)
df9.rename(columns={'region': 'continent'}, inplace=True)

In [ ]:
df9.isna().sum()

In [ ]:
df9.dropna(subset=['continent'], inplace=True)
print(df9.shape)  # Check if the shape has changed

In [ ]:
df9["id"] = range(1, len(df9) + 1)

In [ ]:
new_column_order = ['id', 'name', 'status', 'year', 'place', 'type', 'decade', 'latitude',
       'longitude', 'ccode', 'country', ' sq km', 'continent', 'mass']
df9 = df9[new_column_order]

In [ ]:
df9.to_csv('Full_Final_Dataset.csv', index = False)

# IMPORT FINAL DATASET

In [ ]:
df_final = pd.read_csv('Full_Final_Dataset.csv')

## Rogue NaN value removal

In [ ]:
df_final['year'].value_counts().sort_index(ascending=True)

In [ ]:
df_final['continent'].value_counts().sort_values(ascending = False) # Continental Data for Canva Chart

In [ ]:
df_final['year'].value_counts().sort_index() # Yearly data for Year Canva Chart

In [ ]:
df_final[df_final['continent'] == 'Antarctica']

## Convert and Clean Mass Column, create df_final_mass DF

In [ ]:
df_final['mass'].sort_values()

In [ ]:
df_final['mass'].isna().count()

In [ ]:
df_final.shape

In [ ]:
df_final.dtypes

In [ ]:
df_final['mass'] = pd.to_numeric(df_final['mass'], errors='coerce')

In [ ]:
print(df_final['mass'].dtype)
df_final['mass'] = df_final['mass'].astype('float64')

In [ ]:
df_final['mass'].isnull().sum()


In [ ]:
df_final[df_final['mass'].isnull()]

In [ ]:
df_final.dropna(how='any', inplace=True)

In [ ]:
df_final['mass'].isnull().sum()

In [ ]:
df_final['mass'] = df_final['mass'].astype('int')

In [ ]:
# Drops '0' Values to prevent skewed data - Creates final DF
df_final_mass = df_final[df_final != 0].dropna(how='any')
df_final_mass

In [ ]:
df_final_mass['mass'] = pd.to_numeric(df_final['mass'], errors='coerce')

# DATA ANALYSIS

In [ ]:
# Statistics Function
def statistics(df, column):
    print(f'Statistical Overview:-')
    print(f"Mean: {df[column].mean():.2f}")
    print(f"Median: {df[column].median():.2f}")
    print(f"Mode: {df[column].mode().round(2).values}")
    print(f"Standard Deviation: {df[column].std():.2f}")
    print(f"Variance: {df[column].var():.2f}")
    print(f"Minimum: {df[column].min():.2f}")
    print(f"Maximum: {df[column].max():.2f}")
    print(f"Count: {df[column].count()}")
    print(f'Null Values: {df[column].isna().sum()}')
    return

In [ ]:
statistics(df_final_mass, 'mass')

In [ ]:
df_final_mass.head(20).sort_values(by='mass', ascending=True)


## Cleaning for Type Meteorite Type Analysis

In [ ]:
df_final_mass['type'].value_counts().sort_values(ascending = False).head(10)

In [ ]:
total_nulls = df_final_mass['type'].isna().sum()
print(total_nulls)


In [ ]:
# Strip out random non Alpha-Numeric values
import pandas as pd
import re

df_final_mass['clean_type'] = df_final_mass['type'].str.replace(r'[^\w\s]', '', regex=True)

In [ ]:
df_final_mass['clean_type'].value_counts().sort_values(ascending = False).head(10)

In [ ]:
df_final.columns

## DATA VIS

### Meteorite Mass Scatter

In [ ]:
df_final_mass['type'].value_counts().head(10)

In [ ]:
plt.scatter(df_final_mass.index, df_final_mass['mass'])
plt.xlabel("Count")
plt.ylabel("Mass")
plt.title("Scatter Plot of Meteorite Mass")
plt.show()

### Distribution of Meteorite Mass

In [ ]:
import pandas as pd

# Define the bins for mass ranges (in kilograms, or use appropriate units)
bins = [0, 10, 100, 500, 1000, 5000, 10000]  # Modify the bin ranges as needed
bin_labels = ['0-10 kg', '10-100 kg', '100-500 kg', '500-1000 kg', '1000-5000 kg', '5000-10000 kg']

# Assign each meteorite mass to a bin
df_final_mass['mass_bin'] = pd.cut(df_final_mass['mass'], bins=bins, labels=bin_labels)

# Count how many meteorites fall into each bin
mass_counts = df_final_mass['mass_bin'].value_counts().sort_index()

# Convert the counts to a DataFrame for display
mass_counts_df = mass_counts.reset_index()
mass_counts_df.columns = ['Mass Range', 'Count']

# Display the table
print(mass_counts_df)

# Plotting the bin counts
plt.figure(figsize=(10, 6))
mass_counts.sort_index().plot(kind='bar')

# Add labels and title
plt.xlabel('Mass Ranges')
plt.ylabel('Number of Meteorites')
plt.title('Distribution of Meteorite Masses')

# Show plot
plt.tight_layout()
plt.show()


### Log Scale Meteorite Mass Plot

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import linregress

# Define the bins for mass ranges (in kilograms, or use appropriate units)
bins = [0, 10, 100, 500, 1000, 5000, 10000]  # Modify the bin ranges as needed
bin_labels = ['0-10 kg', '10-100 kg', '100-500 kg', '500-1000 kg', '1000-5000 kg', '5000-10000 kg']

# Assign each meteorite mass to a bin
df_final_mass['mass_bin'] = pd.cut(df_final_mass['mass'], bins=bins, labels=bin_labels)

# Count how many meteorites fall into each bin
mass_counts = df_final_mass['mass_bin'].value_counts().sort_index()

# Calculate midpoints for the mass bins
mass_midpoints = [5, 55, 300, 750, 3000, 7500]

# Convert the counts to a DataFrame for display
mass_counts_df = mass_counts.reset_index()
mass_counts_df.columns = ['Mass Range', 'Count']

# Convert to log scale (log10) for plotting
log_mass = np.log10(mass_midpoints)
log_counts = np.log10(mass_counts_df['Count'])

# Create a log-log plot
plt.figure(figsize=(7, 5))

# Log-log plot
plt.subplot(1, 2, 1)  # Side-by-side with linear plot
plt.scatter(log_mass, log_counts, label="Meteorite Counts", color='blue')
plt.xscale('linear')
plt.yscale('linear')
plt.xlabel("Log(Mass Midpoint) [log10(kg)]")
plt.ylabel("Log(Count) [log10]")
plt.title("Log-Log Scale Plot of Meteorite Mass vs Count")
plt.grid(True)

# Fit a line to the log-log data to check for a power-law relationship
slope, intercept, r_value, p_value, std_err = linregress(log_mass, log_counts)

# Plot the fitted line
fit_label = f"Fit: slope={slope:.2f}, R²={r_value**2:.4f}"
plt.plot(log_mass, slope * log_mass + intercept, 'r-', label=fit_label)
plt.legend()

plt.tight_layout()
plt.show()

# Display the table
print(mass_counts_df)


### Continent Analysis for Canva Charts

In [ ]:
df_final_mass['continent'].sort_values

In [ ]:
df_mass_cont = df_final_mass.groupby("continent")['name'].count().sort_values
df_mass_cont

In [ ]:
df_mass_cont.to_csv('mass_cont.csv', index = False)

In [ ]:
df_mass_country = df_final_mass.groupby("country")['name'].count().sort_values(ascending = False).head(10)
df_mass_country

### Mass v Class Comparison

In [ ]:
df_final_mass.columns

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

bins = [0, 100, 1000, 10000, float('inf')]  # Mass bin thresholds
labels = ['Small Mass (<100g)', 'Medium Mass (100g-1kg)', 'Large Mass (1kg-10kg)', 'Very Large Mass (>10kg)']
df_final_mass['mass_category'] = pd.cut(df_final_mass['mass'], bins=bins, labels=labels, right=False)

# Create a crosstab of meteorite class vs mass category
crosstab_data = pd.crosstab(df_final_mass['clean_type'], df_final_mass['mass_category'])

# Get the top 5 meteorite categories by total count
top_5_categories = crosstab_data.sum(axis=1).nlargest(5).index

# Filter the crosstab to include only the top 5 meteorite categories
crosstab_data_filtered = crosstab_data.loc[top_5_categories]

# Now we transpose the crosstab so that mass categories are on the y-axis and meteorite class on the x-axis
crosstab_data_transposed = crosstab_data_filtered.T

# Reversing the y-axis (mass categories) order
crosstab_data_transposed = crosstab_data_transposed[::-1]

# Generate the heatmap with shades of grey
plt.figure(figsize=(10, 6))
sns.heatmap(crosstab_data_transposed, annot=True, cmap="Greys", fmt="d", cbar_kws={'label': 'Number of Meteorites'})
plt.title("Top 5 Meteorite Categories by Count vs Mass")
plt.xlabel("Meteorite Class")
plt.ylabel("Mass Categories")
plt.show()


In [ ]:
crosstab_data_filtered

# DATA NORMALISATION

In [ ]:
total_count = mass_counts.sum()

normalized_counts = mass_counts / total_count

normalized_counts_df = normalized_counts.reset_index()
normalized_counts_df.columns = ['Mass Range', 'Normalized Count']

print(normalized_counts_df)

In [ ]:
# Plot the normalized distribution
plt.figure(figsize=(7, 5))

plt.bar(mass_counts.index, normalized_counts, color='skyblue')
plt.xlabel("Mass Range [kg]")
plt.ylabel("Normalized Count")
plt.title("Normalized Distribution of Meteorite Mass Ranges")
plt.grid(True)
plt.show()


# PROBABILITY

### Pearson correlation between mass and year

In [ ]:
from scipy import stats


corr, p_value = stats.pearsonr(df_final_mass["year"], df_final_mass["mass"])

print(f"Pearson Correlation Coefficient: {corr:.2f}")
print(f"P-value: {p_value:.4f}")

if p_value < 0.05:
    print("Reject the null hypothesis: There is a significant correlation between meteorite mass and year.")
else:
    print("Fail to reject the null hypothesis: No significant correlation between meteorite mass and year.")


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Create df_heat from df_final_mass
df_heat = df_final_mass.copy()

# Clean the data
df_heat = df_heat.dropna(subset=["year", "mass"])
df_heat["year"] = pd.to_numeric(df_heat["year"], errors="coerce")
df_heat["mass"] = pd.to_numeric(df_heat["mass"], errors="coerce")
df_heat = df_heat.dropna(subset=["year", "mass"])

# Define 50g bins for mass
bin_edges = np.arange(0, df_heat["mass"].max() + 50, 50)  # Create bins of size 50g
df_heat["mass_bin"] = pd.cut(df_heat["mass"], bins=bin_edges, labels=bin_edges[:-1])

# Convert mass_bin to numeric values (not categorical)
df_heat["mass_bin"] = df_heat["mass_bin"].astype(float)

# Create the heatmap plot and assign it to a variable
plt.figure(figsize=(12, 6))
heatmap_plot = sns.histplot(
    data=df_heat, x="year", y="mass_bin", bins=50, cbar=True, cmap="coolwarm"
)

# Fix potential colorbar issue
if heatmap_plot.collections:
    plt.colorbar(heatmap_plot.collections[0], label="Meteorite Count")

# Labels & Title
plt.title("Heatmap of Meteorite Mass vs Year (50g Bins)")
plt.xlabel("Year")
plt.ylabel("Mass Bin (g)")
plt.show()


In [ ]:
from scipy.stats import spearmanr

# Calculate Spearman rank correlation
corr, p_value = spearmanr(df_final_mass["year"], df_final_mass["mass"])

# Print results
print(f"Spearman Rank Correlation Coefficient: {corr:.2f}")
print(f"P-value: {p_value:.4f}")

if p_value < 0.05:
    print("Reject the null hypothesis: There is a significant correlation between meteorite mass and year.")
else:
    print("Fail to reject the null hypothesis: No significant correlation between meteorite mass and year.")


### Chi Squared Test - Correlation between Composition and Size

In [ ]:
import numpy as np
import pandas as pd
import scipy.stats as stats

bins = [0, 1000, 100000, np.inf]
labels = ['Small', 'Medium', 'Large']

df_final_mass['mass_category'] = pd.cut(df_final_mass['mass'], bins=bins, labels=labels)

contingency_table = pd.crosstab(df_final_mass['type'], df_final_mass['mass_category'])

# Chi-Square Test
chi2_stat, p_value, dof, expected = stats.chi2_contingency(contingency_table)

print(f"Chi-Square Test Statistic: {chi2_stat:.4f}, p-value: {p_value:.4f}")
print("Degrees of Freedom:", dof)

if p_value < 0.05:
    print("There is a significant relationship between meteorite mass category and composition.")
else:
    print("No significant relationship between meteorite mass category and composition.")

print("\nObserved Counts:")
print(contingency_table)

print("\nExpected Counts (if no relationship existed):")
print(pd.DataFrame(expected, index=contingency_table.index, columns=contingency_table.columns))



In [ ]:
import pandas as pd
import scipy.stats as stats

bins = [0, 100, 1000, 10000, float('inf')]  # Mass bin thresholds
labels = ['Small Mass (<100g)', 'Medium Mass (100g-1kg)', 'Large Mass (1kg-10kg)', 'Very Large Mass (>10kg)']

df_final_mass['mass_category'] = pd.cut(df_final_mass['mass'], bins=bins, labels=labels, right=False)

contingency_table = pd.crosstab(df_final_mass['mass_category'], df_final_mass['type'])

chi2_stat, p_val, dof, expected = stats.chi2_contingency(contingency_table)

print("Contingency Table:")
print(contingency_table)
print("\nChi-Squared Statistic:", chi2_stat)
print("p-value:", p_val)
print("Degrees of Freedom:", dof)
print("\nExpected Frequencies:")
print(expected)


alpha = 0.05
if p_val < alpha:
    print("\nResult: There is a significant relationship between mass category and meteorite type (reject null hypothesis).")
else:
    print("\nResult: There is no significant relationship between mass category and meteorite type (fail to reject null hypothesis).")


In [ ]:
print(df_final_mass['type'].unique())

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Define Mass Bins (from chi-squared test)
mass_bins = [0, 100, 1000, 10000, float("inf")]  
mass_labels = ["Small Mass (<100g)", "Medium Mass (100g-1kg)", "Large Mass (1kg-10kg)", "Very Large Mass (>10kg)"]

# Categorization Function for Meteorite Type
def categorize_meteorite(meteorite_type):
    meteorite_type = str(meteorite_type).lower()  # Normalize case
    if any(x in meteorite_type for x in ["h", "l", "ll", "eh", "el", "ck", "co", "cm", "cv", "cr", "r"]):
        return "Chondrite (Stony)"
    elif any(x in meteorite_type for x in ["eucrite", "diogenite", "howardite", "ureilite", "aubrite", "angrite", "lunar", "martian"]):
        return "Achondrite"
    elif "iron" in meteorite_type:
        return "Iron"
    elif any(x in meteorite_type for x in ["pallasite", "mesosiderite"]):
        return "Stony-Iron"
    else:
        return "Other"

# Apply Meteorite Type Categorization
df_final_mass["Type Category"] = df_final_mass["type"].apply(categorize_meteorite)

# Apply Mass Binning
df_final_mass["Mass Category"] = pd.cut(df_final_mass["mass"], bins=mass_bins, labels=mass_labels, right=False)

# Create Contingency Table
contingency_table = df_final_mass.pivot_table(index="Mass Category", columns="Type Category", aggfunc="size", fill_value=0)

# Reorder index to ensure smallest mass is at the bottom
contingency_table = contingency_table.reindex(mass_labels[::-1])  # Reverse order

# Plot Heatmap with Greyscale Color Map
plt.figure(figsize=(10, 6))
sns.heatmap(contingency_table, annot=True, cmap="Greys", linewidths=0.5, fmt="d")
plt.title("Meteorite Mass vs Type Heatmap")
plt.xlabel("Meteorite Type")
plt.ylabel("Mass Category")
plt.xticks(rotation=30, ha="right")  # Rotate for better readability
plt.show()
